#### **Library importation**

In [267]:
!pip install optuna
!pip install alibi
!pip install lime

In [93]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [95]:
import numpy as np
import pandas as pd
import optuna
from sklearn.preprocessing import MinMaxScaler
import xgboost

from utils import encode_features, get_train_test_data, train_model, evaluate_model, generate_individual, epsilon_rounding, get_relevant_candidates

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Loan counterfactuals

In [249]:
def load_data(data_filepath="/content/drive/MyDrive/data/Loan_data_extracted.csv"):
    """
    Input: path to .csv data file

    TODO: specify in feature_info whether features are of type:
        fixed, meaning cannot change for the counterfactual
        unique, meaning can only take existing categorical values
        increase, meaning their value can only increase and not decrease
        range, meaning their new value can take a range of values

    Returns:
        dataframe and feature configuration dictionary
    """
    df = pd.read_csv(data_filepath)
    df = df.drop('Loan_ID', axis=1)
    df = df.dropna()

    feature_config = {
        "categorical": ["Gender", "Married", "Education", "Self_Employed", "Property_Area", "Loan_Status"],
        "feature_info": [
            ('Gender', 'fixed'),
            ('Married', 'fixed'),
            ('Dependents', 'unique'),
            ('Education', 'fixed'),
            ('Self_Employed', 'unique'),
            ('ApplicantIncome', 'increase'),
            ('CoapplicantIncome', 'increase'),
            ('LoanAmount', 'range'),
            ('Loan_Amount_Term', 'unique'),
            ('Credit_History', 'unique'),
            ('Property_Area', 'fixed'),
        ],


        "categorical_features": ["Gender", "Married", "Education", "Self_Employed", "Property_Area"]
    }

    return df, feature_config

In [250]:
df, feature_config =load_data(data_filepath="/content/drive/MyDrive/data/Loan_data_extracted.csv")
df.head(3)

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,No,0,Graduate,No,3676,4301,172.0,360,1,Rural,Y
1,Female,Yes,1,Graduate,No,12000,0,496.0,360,1,Semiurban,Y
3,Male,Yes,1,Graduate,No,3400,2500,173.0,360,1,Semiurban,Y


In [251]:
df.nunique()

,0
Gender,2
Married,2
Dependents,4
Education,2
Self_Employed,2
ApplicantIncome,10
CoapplicantIncome,6
LoanAmount,10
Loan_Amount_Term,2
Credit_History,2


In [252]:
# Load the model from the saved file
model = xgboost.XGBClassifier()
model.load_model("xgboost_model.json")

In [253]:
def misfit(x_prime, y_target, model):
    """
    Optimisation criterion 1
    Calculate absolute difference between y_target and y_prime_prediction.
    """

    #TODO
    y_prime_prediction = model.predict_proba(x_prime)[0][1]
    abs_diff = np.abs(y_prime_prediction - y_target)
    return float(abs_diff)

In [254]:
def distance(X, x, x_prime, numerical, categorical):
    """
    Optimisation criterion 2
    Calculate distance between x_prime and x.
    """
    # Normalize data
    scaler = MinMaxScaler()
    scaler.fit(X[numerical])
    x_normalized = scaler.transform(x[numerical])
    x_prime_normalized = scaler.transform(x_prime[numerical])

    # Compute distances
    #TODO
    num_distance = np.abs(x_prime_normalized - x_normalized).sum()
    cat_distance  = (x[categorical] != x_prime[categorical]).sum(axis = 1).iloc[0]
    return (num_distance + cat_distance)/x_prime.shape[1]


In [255]:
def sparsity(x, x_prime):
    """
    Optimisation criterion 3
    Return number of unchanged features.
    """
    #TODO
    return (x == x_prime).sum(axis = 1).iloc[0]


In [256]:
def closest_real(X, x_prime, categorical, numerical):
    """
    Optimisation criterion 4
    Return the minimum distance between x_prime and any point in X.
    """
    scaler = MinMaxScaler()
    X_normalized = scaler.fit_transform(X[numerical])
    x_prime_normalized = scaler.transform(x_prime[numerical])

    # Compute total distance
    #TODO
    obs_number = X_normalized.shape[0]
    min = np.inf
    for i in range(obs_number):
       num_distance = np.abs(X_normalized[i,:] - x_prime_normalized).sum()
       cat_distance = (X.iloc[i][categorical] != x_prime[categorical]).sum(axis = 1).iloc[0]
       distance = num_distance + cat_distance
       if distance < min :
        min = distance
    features_number = x_prime.shape[1]
    return float(min/features_number)

In [257]:
def objective(trial, X, x, features, model, y_target, numerical, categorical):
    x_prime = x.copy()
    for feature in features:
        feature.sample(trial)
        x_prime[feature.name] = feature.value
    epsilon_rounding(x, x_prime, 1e-1)

    obj1 = misfit(x_prime, y_target, model)
    obj2 = distance(X, x, x_prime, numerical, categorical)
    obj3 = sparsity(x, x_prime)
    obj4 = closest_real(X, x_prime, categorical, numerical)

    return obj1, obj2, obj3, obj4


In [258]:
def get_counterfactuals(X, x, y_target, model,
                        numerical, categorical, features,
                        tol, optimization_steps, timeout):

    study = optuna.create_study(directions=['minimize', 'minimize', 'maximize', 'minimize'],
                                sampler=optuna.samplers.NSGAIISampler(seed=42))

    study.optimize(lambda trial: objective(trial, X, x, features, model,
                                           y_target,
                                           numerical,
                                           categorical),
                   n_trials=optimization_steps,
                   timeout=timeout)

    candidates_df = get_relevant_candidates(study, x, model, y_target, tol)

    return candidates_df

In [259]:
# Check that our customer x did not get the loan
X_obs, feat_conf = load_data(data_filepath="/content/drive/MyDrive/data/Loan_data_extracted.csv")
X_obs = encode_features(X_obs, feat_conf["categorical"])
customer = np.array([0,1,0,0,0,2000,1500,1000,480,0,1])
x = pd.DataFrame([customer], columns=X_obs.columns[:-1].tolist())
x


--- 
Encoded categorical features as follows:
Gender :  {'Female': 0, 'Male': 1}
Married :  {'No': 0, 'Yes': 1}
Education :  {'Not Graduate': 0, 'Graduate': 1}
Self_Employed :  {'No': 0, 'Yes': 1}
Property_Area :  {'Rural': 0, 'Semiurban': 1, 'Urban': 2}
Loan_Status :  {'N': 0, 'Y': 1}
---


,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,0,1,0,0,0,2000,1500,1000,480,0,1


In [260]:
print(f"Probability of original instance (x): {model.predict_proba(x)[:, 1]}")

Probability of original instance (x): [0.00302464]


In [261]:
# TODO
y_obs_pred = model.predict(x)
print(f"{y_obs_pred}")
# and help her find out what she has to do in order to get the loan
# If you have implemented everything above correctly, the code below
# will find the counterfactuals

[0]


In [262]:
# Make a list of Feature objects containing information about how
# each feature is allowed to change when generating counterfactuals
change_features = generate_individual(X_obs, x, feat_conf["feature_info"])

In [263]:
# Set the desired new model prediction
y_CF = 0.7
print(f"Searching for counterfactuals with y_CF = {y_CF}...\n")
numerical_features = [x for x in df.columns if x not in feat_conf["categorical"]]
CFS = get_counterfactuals(X_obs, x, y_CF, model,
                          numerical_features,
                          feat_conf["categorical_features"],
                          change_features,
                          tol=0.05,
                          optimization_steps=500,
                          timeout=None)

Searching for counterfactuals with y_CF = 0.7...



In [264]:
x

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,0,1,0,0,0,2000,1500,1000,480,0,1


In [265]:
CFS

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,0.0,1.0,0.0,0.0,0.0,2000.0,1500.0,173.877881,360.0,1.0,1.0


In [266]:
print(f"Probability of original instance (x): {model.predict_proba(x)[:, 1]}")
print(f"Probability of counterfactuals (CFS): {model.predict_proba(CFS)[:, 1]}")

Probability of original instance (x): [0.00302464]
Probability of counterfactuals (CFS): [0.25187543]


We can say that it would be difficult for this client to obtain this loan. Their probability of getting the loan was close to zero, and we were able to increase it to 0.25187543 with our analysis. This also depends on the fact that, technically, we fixed several variables to avoid discrimination, as well as the number of iterations.